## Using the trained model to predict in our dataset

### First we import the packages and the dataset

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

import pandas as pd

# Start by importing the test data
test_path = Path('data/raw/dataset_test.csv')
df_test = pd.read_csv(test_path)

Working dir: c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting


### Now we perform the datacleaning

In [2]:
from src.data_cleaning import clean_data

df_test = clean_data(dataset= df_test, is_train = False, categorize_station=True)

df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 537445 entries, 0 to 537444
Data columns (total 22 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   id                     537445 non-null  str           
 1   datetime               537445 non-null  datetime64[us]
 2   station_number         537445 non-null  category      
 3   name                   537445 non-null  str           
 4   lat                    537445 non-null  float64       
 5   lng                    537445 non-null  float64       
 6   hour                   537445 non-null  int32         
 7   minute                 537445 non-null  int32         
 8   dayofweek              537445 non-null  int32         
 9   month                  537445 non-null  int32         
 10  is_weekend             537445 non-null  int64         
 11  hour_sin               537445 non-null  float64       
 12  hour_cos               537445 non-null  float64       


Spot-check rows at midnight to visually confirm the cleaning and feature engineering look correct.

In [3]:
df_test[df_test['hour'] == 0]

,id,datetime,station_number,name,lat,lng,hour,minute,dayofweek,month,...,hour_cos,is_holiday,avg_bikes_per_station,temperature_2m,apparent_temperature,precipitation,snowfall,wind_speed_10m,cloud_cover,relative_humidity_2m
7285,2025-03-14 00:00:00_32000,2025-03-14 00:00:00,32000,Julius-Raab-Platz,48.211544,16.382374,0,0,4,3,...,1.0,0,9.258033,4.4,2.1,0.0,0.0,4.1,100,87
7286,2025-03-14 00:00:00_32001,2025-03-14 00:00:00,32001,Hoher Markt,48.210666,16.372983,0,0,4,3,...,1.0,0,16.898142,4.4,2.1,0.0,0.0,4.1,100,87
7287,2025-03-14 00:00:00_32002,2025-03-14 00:00:00,32002,Oper,48.202683,16.369702,0,0,4,3,...,1.0,0,14.598361,4.4,2.1,0.0,0.0,4.1,100,87
7288,2025-03-14 00:00:00_32003,2025-03-14 00:00:00,32003,Volksgarten,48.206516,16.360400,0,0,4,3,...,1.0,0,7.415082,4.4,2.1,0.0,0.0,4.1,100,87
7289,2025-03-14 00:00:00_32004,2025-03-14 00:00:00,32004,Taborstraße U2,48.219522,16.382218,0,0,4,3,...,1.0,0,12.433880,4.4,2.1,0.0,0.0,4.1,100,87
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526630,2025-04-29 00:30:00_32275,2025-04-29 00:30:00,32275,eLastenräder - Am langen Felde,48.250224,16.450650,0,30,1,4,...,1.0,0,0.675519,11.6,10.5,0.0,0.0,5.2,1,83
526631,2025-04-29 00:30:00_32277,2025-04-29 00:30:00,32277,eLastenräder - Bruno-Marek-Allee 6,48.227914,16.391516,0,30,1,4,...,1.0,0,3.513661,11.6,10.5,0.0,0.0,5.2,1,83
526632,2025-04-29 00:30:00_32278,2025-04-29 00:30:00,32278,eLastenräder - Am Tabor 23,48.224598,16.392090,0,30,1,4,...,1.0,0,2.462186,11.6,10.5,0.0,0.0,5.2,1,83
526633,2025-04-29 00:30:00_32280,2025-04-29 00:30:00,32280,ALF Mobility-Point,48.251355,16.452810,0,30,1,4,...,1.0,0,4.515519,11.6,10.5,0.0,0.0,5.2,1,83


Keep the `id` column (needed for the submission file) and drop identifier/non-predictive columns before running inference.

In [3]:
# Save IDs and do preprocessing (as we did before)

test_ids = df_test["id"].copy()      # save the ids

DROP_COLS = ["id", "datetime", "name"]#, 

df_test = df_test.drop(columns = DROP_COLS)


In [13]:
test_ids

0         2025-03-13 08:30:00_32000
1         2025-03-13 08:30:00_32001
2         2025-03-13 08:30:00_32002
3         2025-03-13 08:30:00_32003
4         2025-03-13 08:30:00_32004
                    ...            
537440    2025-04-29 23:30:00_32275
537441    2025-04-29 23:30:00_32277
537442    2025-04-29 23:30:00_32278
537443    2025-04-29 23:30:00_32280
537444    2025-04-29 23:30:00_32283
Name: id, Length: 537445, dtype: str

### Do the predictions

Load the trained model, predict bike counts for the test set, then clip negative predictions to 0 and round to whole numbers (bike counts must be non-negative integers).

In [4]:
import joblib
import pandas as pd
import numpy as np

# Load the trained model
model_path = Path('models/lightgbmv6.joblib')
model = joblib.load(model_path)

# Make predictions
predictions = model.predict(df_test)

#rund the predictions
predictions = np.maximum(0, predictions).round().astype(int)


### Prepare the submission

In [5]:
submission = pd.DataFrame({"id": test_ids, "bikes": predictions})

# and now save the predictions
from pathlib import Path

out_path = Path('reports/predictions.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)